# MSTAR Classification with ResNet-50

Training a ResNet-50 model on the MSTAR dataset (SAR images of military vehicles) to establish a baseline for adversarial attack evaluation.

**Dataset:** 8 vehicle classes, ~9,500 images  
**Target:** >90% test accuracy

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
from tqdm.auto import tqdm
import kagglehub

# Device: CUDA > MPS > CPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'CUDA: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

In [ ]:
DATA_DIR = Path('Padded_imgs')

if DATA_DIR.exists() and any(DATA_DIR.iterdir()):
    print(f'Dataset already exists at {DATA_DIR}')
else:
    print('Downloading MSTAR dataset from Kaggle...')
    download_path = Path(kagglehub.dataset_download('atreyamajumdar/mstar-dataset-8-classes'))
    
    # kagglehub extracts to a versioned folder - find Padded_imgs
    if (download_path / 'Padded_imgs').exists():
        src = download_path / 'Padded_imgs'
    else:
        src = download_path
    
    shutil.copytree(src, DATA_DIR, dirs_exist_ok=True)
    print(f'Dataset downloaded to {DATA_DIR}')

print(f'Dataset directory: {DATA_DIR.resolve()}')

In [ ]:
# Configuration
CHECKPOINT_DIR = Path('checkpoints')
CHECKPOINT_DIR.mkdir(exist_ok=True)

BATCH_SIZE  = 64
EPOCHS      = 50
LR          = 0.001
NUM_CLASSES = 8
RESUME      = True  # Resume from checkpoint if exists

# Workers: scale with available CPU cores on cluster
if device.type == 'cuda':
    NUM_WORKERS = min(8, os.cpu_count() or 4)
else:
    NUM_WORKERS = 0
PIN_MEMORY = device.type == 'cuda'

print(f'Batch size: {BATCH_SIZE}')
print(f'Workers: {NUM_WORKERS}')
print(f'Resume: {RESUME}')

## Data Loading

In [ ]:
# ImageNet normalization for pretrained ResNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
class_names = full_dataset.classes
print(f'Classes: {class_names}')
print(f'Total images: {len(full_dataset)}')

In [ ]:
# 70% train, 15% val, 15% test
n_total = len(full_dataset)
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

# No augmentation for val/test
val_dataset.dataset.transform = test_transform
test_dataset.dataset.transform = test_transform

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

## Dataset Exploration

In [ ]:
# Class distribution
class_counts = {}
for cls in class_names:
    class_counts[cls] = len(list((DATA_DIR / cls).glob('*.JPG'))) + len(list((DATA_DIR / cls).glob('*.jpg')))

plt.figure(figsize=(10, 5))
plt.bar(class_counts.keys(), class_counts.values())
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.title('MSTAR Class Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Sample images from each class
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return tensor * std + mean

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

samples_per_class = {}
for img, label in full_dataset:
    if label not in samples_per_class:
        samples_per_class[label] = img
    if len(samples_per_class) == NUM_CLASSES:
        break

for idx, (label, img) in enumerate(sorted(samples_per_class.items())):
    img_display = denormalize(img).permute(1, 2, 0).clamp(0, 1).numpy()
    axes[idx].imshow(img_display)
    axes[idx].set_title(class_names[label])
    axes[idx].axis('off')

plt.suptitle('Sample Images from Each Class')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150)
plt.show()

## Model Setup

In [ ]:
# Pretrained ResNet-50 - weights auto-download on first run
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## Training

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    
    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    
    return total_loss / total, correct / total

In [ ]:
# Training loop with checkpointing and resume support
RESUME_CKPT = CHECKPOINT_DIR / 'resume_checkpoint.pth'

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 10
start_epoch = 0

# Resume from checkpoint if exists
if RESUME and RESUME_CKPT.exists():
    print(f'Resuming from {RESUME_CKPT}')
    ckpt = torch.load(RESUME_CKPT, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt['best_val_acc']
    patience_counter = ckpt['patience_counter']
    history = ckpt['history']
    print(f'Resumed at epoch {start_epoch}, best val acc: {best_val_acc:.4f}')
else:
    print('Starting fresh training run')

# Training loop
for epoch in range(start_epoch, EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader)
    
    scheduler.step(val_acc)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_DIR / 'best_model.pth')
        print(f'  -> New best model (val_acc: {val_acc:.4f})')
    else:
        patience_counter += 1
    
    # Save resume checkpoint every epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_acc': best_val_acc,
        'patience_counter': patience_counter,
        'history': history,
        'class_names': class_names,
        'train_indices': train_dataset.indices,
        'val_indices': val_dataset.indices,
        'test_indices': test_dataset.indices,
    }, RESUME_CKPT)
    
    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f'Early stopping at epoch {epoch+1}')
        break

print(f'\nBest validation accuracy: {best_val_acc:.4f}')

## Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves')
ax1.legend()

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curves')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## Final Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(CHECKPOINT_DIR / 'best_model.pth', map_location=device))
test_loss, test_acc = evaluate(model, test_loader)
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')

In [ ]:
# Confusion matrix
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix (Test Accuracy: {test_acc:.2%})')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
print('Classification Report:')
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Save dataset info for eval notebook
torch.save({
    'test_indices': test_dataset.indices,
    'class_names': class_names,
    'test_accuracy': test_acc
}, CHECKPOINT_DIR / 'dataset_info.pth')

print(f'Saved model and dataset info to {CHECKPOINT_DIR}/')